# Notebook 09a — Robustness I: cheap sweeps (prior dispersion $\sigma_0^2$ and count-variance window $W$)

*Thesis §Model Robustness, items 1–2.*

Both parameters swept here affect only the **estimation**, not the routing: the two class
incidence matrices $A_{low}$, $A_{high}$ are frozen (from Notebook 05), so each setting re-runs
the whole estimation cheaply — Stage-1 GLS on $A_{neutral}$, then the Stage-3 outer loop — to give
a fresh $\pi^*$ and $t^*$. This is the fast robustness notebook; the slow one that re-routes is 09b.

**Decisions use training data only.** For every setting we report (a) the training mean GEH at that
setting's own $\pi^*$ and (b) a 5-fold training-only cross-validated mean GEH (each fold refits $\pi$
and $t$ on the other folds). Held-out stations are scored once per setting at that setting's frozen
$\pi^*$ and are **reported only** — so $\sigma_0^2$ and $W$ are chosen from training evidence alone.

A baseline-reproduction assertion checks that the baseline row returns the frozen Stage-3 $\pi^*$
from Notebook 06, guaranteeing the sweep is anchored to the main run.

**Inputs:** `A_low.npz`, `A_high.npz`, `A_neutral.npz`, `od_calibrated_t.csv`, `A_neutral_links.csv`,
`count_link_snap.csv`, `train_mask.npy`, `stage3_final.json`, plus the CoS count survey.
**Outputs:** `robustness_sigma2.csv`, `robustness_wwindow.csv`, `09a_sweeps.png`,
`reports/09a_robustness_report.txt`.


In [6]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
import thesis_common as tc

tc.set_style()
NB = "09a"

# Sweep grids. SIGMA2_SWEEP brackets the baseline tc.SIGMA2; W_WINDOWS are (start, end) year
# ranges handed to tc.build_W — the first entry is the baseline window used by the main run.
SIGMA2_SWEEP = [1.0, 2.0, 3.0, 5.0, 7.5, 10.0]
W_WINDOWS = {"2022-2025 (base)": (2022, 2025), "2022-2023": (2022, 2023), "2023-2024": (2023, 2024),
             "2024-2025": (2024, 2025), "2022-2024": (2022, 2024), "2023-2025": (2023, 2025)}
RUN_CV = True        # 5-fold training-only CV per setting (~1 min per setting); set False to skip


In [7]:
# Load the frozen main-run artefacts: routing matrices, calibrated demand, monitored links,
# the count survey, and the training mask + CV folds.
final = tc.read_final()
A_low  = sparse.load_npz(tc.DATA_PROCESSED / "A_low.npz").tocsr()
A_high = sparse.load_npz(tc.DATA_PROCESSED / "A_high.npz").tocsr()
A_neu  = sparse.load_npz(tc.DATA_PROCESSED / "A_neutral.npz").tocsr()
cal    = pd.read_csv(tc.DATA_PROCESSED / "od_calibrated_t.csv").sort_values("pair_idx").reset_index(drop=True)
links  = pd.read_csv(tc.DATA_PROCESSED / "A_neutral_links.csv").sort_values("link_idx").reset_index(drop=True)
snap   = pd.read_csv(tc.DATA_PROCESSED / "count_link_snap.csv")
L = len(links); v_obs = links.v_obs.to_numpy(float); n_seed = cal.n_seed.to_numpy(float)
train = tc.load_train_mask(L)
folds = tc.make_cv_folds(train, v_obs)
survey = tc.load_survey(); s2l = tc.site_to_link(snap, links)
print(f"train {int(train.sum())} | CV folds {[int(f_.sum()) for f_ in folds]} | baseline pi* (NB07) {final['pi_star']:.2f}")

def run_setting(W, sigma2):
    """Re-estimate the whole model for one (W, sigma2) setting on the training stations.

    Stage-1 GLS -> Stage-3 outer loop -> pi*, t*. Returns the training-fit metrics, the
    training-only CV mean GEH (if RUN_CV), and the fitted flow vector for the held-out read.
    """
    t1 = tc.solve_gls(A_neu, n_seed, v_obs, W, train, sigma2)                 # Stage 1
    r = tc.outer_loop(A_low, A_high, t1, n_seed, v_obs, W, train, sigma2)     # Stage 3
    f = tc.flow(A_low, A_high, r["pi_star"], r["t_star"])
    m = tc.metrics(f[train], v_obs[train])
    out = dict(pi_star=r["pi_star"], n_iter=r["n_iter"], converged=r["reason"], t_sum=float(r["t_star"].sum()),
               train_mean_geh=m["mean_geh"], train_geh_pass=m["geh_pass"], train_rmse=m["rmse"], train_r2=m["r2"],
               pi_rmse=tc.objective_curves(A_low, A_high, r["t_star"], v_obs, train)["pi_rmse"])
    if RUN_CV:
        out["cv_mean_geh"], _, pis = tc.cv_score(A_low, A_high, t1, n_seed, v_obs, W, train, folds, sigma2)
        out["cv_pi_range"] = f"{min(pis):.2f}-{max(pis):.2f}"
    return out, f

def heldout(f, purpose):
    """Single held-out evaluation for one setting (reported only, never used to choose)."""
    test = tc.load_test_mask(L, purpose, "NB09a")
    m = tc.metrics(f[test], v_obs[test])
    return dict(test_mean_geh=m["mean_geh"], test_geh_pass=m["geh_pass"], test_rmse=m["rmse"], test_r2=m["r2"])


train 53 | CV folds [11, 11, 11, 10, 10] | baseline pi* (NB07) 0.64


## Sweep 1 — prior dispersion $\sigma_0^2$

$\sigma_0^2$ is the variance of the GLS prior $Q=\sigma_0^2 I$: how far the calibrated demand $t$ is
allowed to drift from the census seed $n$. It has no literature value, so this sweep is the evidence
behind the baseline choice — the training-only CV picks the preferred value.


In [8]:
# Sweep sigma0^2 at the baseline W. Each row re-estimates the model and records training fit,
# CV fit, and the (reported-only) held-out fit.
W_base, _ = tc.build_W(survey, s2l, L)
rows = []
for s2 in SIGMA2_SWEEP:
    r, f = run_setting(W_base, s2)
    r.update(heldout(f, f"robustness sigma2={s2}"))                       # reported only
    r.update(param="sigma2", value=s2, label=f"{s2:g}", is_baseline=(s2 == tc.SIGMA2)); rows.append(r)
    print(f"  sigma0^2={s2:>5}: pi*={r['pi_star']:.2f}  train meanGEH {r['train_mean_geh']:.2f}  "
          f"CV meanGEH {r.get('cv_mean_geh', np.nan):.2f}  | held-out R2 {r['test_r2']:.2f} (reported only)")
sig = pd.DataFrame(rows)

# Guard: the baseline row must reproduce the frozen Stage-3 pi* exactly, or the inputs are out of sync.
base = sig[sig.is_baseline].iloc[0]
assert abs(base.pi_star - final["pi_star"]) < 1e-9, (
    f"baseline sigma0^2 run gives pi*={base.pi_star:.2f} but NB07 froze {final['pi_star']:.2f} - inputs out of sync")
print("baseline reproduces NB07 pi* exactly")
if RUN_CV:
    best = sig.loc[sig.cv_mean_geh.idxmin()]
    print(f"training-only CV prefers sigma0^2 = {best.value:g} (CV mean GEH {best.cv_mean_geh:.2f}; "
          f"baseline {base.cv_mean_geh:.2f}). Use this as the data-driven support for the baseline choice.")


  sigma0^2=  1.0: pi*=0.67  train meanGEH 6.41  CV meanGEH 7.45  | held-out R2 0.55 (reported only)
  sigma0^2=  2.0: pi*=0.63  train meanGEH 6.08  CV meanGEH 7.35  | held-out R2 0.55 (reported only)
  sigma0^2=  3.0: pi*=0.60  train meanGEH 5.84  CV meanGEH 7.31  | held-out R2 0.55 (reported only)
  sigma0^2=  5.0: pi*=0.62  train meanGEH 5.49  CV meanGEH 7.27  | held-out R2 0.55 (reported only)
  sigma0^2=  7.5: pi*=0.63  train meanGEH 5.20  CV meanGEH 7.29  | held-out R2 0.55 (reported only)
  sigma0^2= 10.0: pi*=0.64  train meanGEH 4.98  CV meanGEH 7.25  | held-out R2 0.55 (reported only)
baseline reproduces NB07 pi* exactly
training-only CV prefers sigma0^2 = 10 (CV mean GEH 7.25; baseline 7.25). Use this as the data-driven support for the baseline choice.


## Sweep 2 — count-variance window $W$

$W$ is the per-link count dispersion used by GLS, estimated as the sample variance of the round-mean
peak counts over a window of survey years. Shrinking or shifting the window changes which rounds feed
that variance, so this sweep shows how sensitive $\pi^*$ is to the choice of window.


In [9]:
# Sweep the count-variance window at the baseline sigma0^2. Each window rebuilds W, re-estimates
# the model, and records the same columns; the baseline window must again reproduce the frozen pi*.
rows = []
for lab, yrs in W_WINDOWS.items():
    Wd, n_own = tc.build_W(survey, s2l, L, years=yrs)
    r, f = run_setting(Wd, tc.SIGMA2)
    r.update(heldout(f, f"robustness W={lab}"))
    r.update(param="W", value=lab, label=lab, n_own_variance=n_own, is_baseline=(yrs == tc.W_YEARS)); rows.append(r)
    print(f"  {lab:<17} ({n_own}/{L} own var): pi*={r['pi_star']:.2f}  train {r['train_mean_geh']:.2f}  "
          f"CV {r.get('cv_mean_geh', np.nan):.2f}")
wdf = pd.DataFrame(rows)
assert abs(wdf[wdf.is_baseline].pi_star.iloc[0] - final["pi_star"]) < 1e-9


  2022-2025 (base)  (77/77 own var): pi*=0.64  train 4.98  CV 7.25
  2022-2023         (77/77 own var): pi*=0.55  train 4.91  CV 7.19
  2023-2024         (77/77 own var): pi*=0.57  train 4.40  CV 7.04
  2024-2025         (77/77 own var): pi*=0.68  train 4.33  CV 6.83
  2022-2024         (77/77 own var): pi*=0.60  train 4.91  CV 7.21
  2023-2025         (77/77 own var): pi*=0.68  train 4.44  CV 6.95


## Save, figures, and report

The $\sigma_0^2$ and $W$-window rows of the robustness table (thesis Table 4.3) are written to
`robustness_sigma2.csv` and `robustness_wwindow.csv`, drawn in `09a_sweeps.png`, and summarised in
`reports/09a_robustness_report.txt`. The report block at the end is this notebook's key-results
printout.


In [10]:
# Save the two sweep tables, draw the twin-axis figure (pi* vs training/CV fit), and write the
# text report. The printed report is this notebook's end-of-run key-results summary.
sig.to_csv(tc.DATA_PROCESSED / "robustness_sigma2.csv", index=False)
wdf.to_csv(tc.DATA_PROCESSED / "robustness_wwindow.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, d, xl, cat in [(axes[0], sig, r"$\sigma_0^2$", False), (axes[1], wdf, "count-variance window", True)]:
    x = np.arange(len(d)) if cat else d.value.astype(float)
    ax.plot(x, d.pi_star, "o-", color="black", label=r"$\pi^*$ (left)")
    ax.axhline(final["pi_star"], color=tc.COL["low"], ls="--", lw=1)
    ax.set_ylim(0, 1); ax.set_ylabel(r"$\pi^*_{low}$"); ax.set_xlabel(xl)
    if cat: ax.set_xticks(x); ax.set_xticklabels(d.label, rotation=25, ha="right")
    axr = ax.twinx(); axr.spines["right"].set_visible(True)
    axr.plot(x, d.train_mean_geh, "s--", color=tc.COL["train"], label="train mean GEH")
    if "cv_mean_geh" in d: axr.plot(x, d.cv_mean_geh, "^:", color=tc.COL["train"], alpha=0.6, label="train-CV mean GEH")
    axr.set_ylabel("mean GEH (training data)", color=tc.COL["train"])
    h = ax.get_legend_handles_labels()[0] + axr.get_legend_handles_labels()[0]
    ax.legend(handles=h, fontsize=8, loc="lower right")
fig.suptitle("Frozen-routing sweeps: decisions use training data only", y=1.03)
tc.save_diag(fig, NB, "sweeps")

cols = ["label", "pi_star", "n_iter", "train_mean_geh", "cv_mean_geh", "train_geh_pass", "test_mean_geh", "test_geh_pass", "test_r2"]
rep = f"""09a - robustness (frozen routing) ({tc.RUN_STAMP})
{'='*60}
Each setting: Stage-1 GLS + Stage-3 outer loop on training stations; CV = 5-fold within training.
Held-out columns are a single evaluation per setting at its frozen pi* (reported, never used to choose).

SIGMA0^2  (pi* span {np.ptp(sig.pi_star):.2f})
{sig[[c for c in cols if c in sig]].round(3).to_string(index=False)}

W WINDOW  (pi* span {np.ptp(wdf.pi_star):.2f})
{wdf[[c for c in cols if c in wdf]].round(3).to_string(index=False)}
"""
(tc.REPORTS / "09a_robustness_report.txt").write_text(rep, encoding="utf-8"); print(rep)


  saved diagnostic     09a_sweeps.png
09a - robustness (frozen routing) (20261028_0730)
Each setting: Stage-1 GLS + Stage-3 outer loop on training stations; CV = 5-fold within training.
Held-out columns are a single evaluation per setting at its frozen pi* (reported, never used to choose).

SIGMA0^2  (pi* span 0.07)
label  pi_star  n_iter  train_mean_geh  cv_mean_geh  train_geh_pass  test_mean_geh  test_geh_pass  test_r2
    1     0.67       5           6.410        7.455          49.057          5.710         37.500    0.554
    2     0.63       5           6.076        7.347          50.943          5.531         41.667    0.553
    3     0.60       5           5.843        7.313          52.830          5.427         41.667    0.552
    5     0.62       5           5.487        7.274          56.604          5.486         41.667    0.553
  7.5     0.63       7           5.197        7.293          60.377          5.545         41.667    0.550
   10     0.64       8           4.977  